# UACB-604 · Lecture 1.2 — A transcriptome is a count matrix

> **Colab users:** this notebook uses **R**. Choose **Runtime → Change runtime type → R** before running.

**Data:** RNA-seq of colorectal cancer, 5 metastasised tumours (`CASE`) and 5 controls (`CTRL`), study SRP029880.
It is the same dataset used in Akalin, *Computational Genomics with R*, chapter 8.

**You will:**
1. Load a real count matrix and look at its shape
2. See why library size matters and compute CPM and TPM
3. Check the lecture's claim that counts are **over-dispersed** (variance > mean)
4. Use PCA to see whether tumour and control samples separate

Only base R is used, so nothing needs installing.

In [ ]:
base <- "https://raw.githubusercontent.com/compgenomr/compGenomRData/master/inst/extdata/rna-seq/"
counts_all <- as.matrix(read.table(paste0(base, "SRP029880.raw_counts.tsv"), header = TRUE, sep = "\t"))
coldata    <- read.table(paste0(base, "SRP029880.colData.tsv"), header = TRUE, sep = "\t")

gene_length <- counts_all[, "width"]           # last column holds gene length (bp)
counts      <- counts_all[, colnames(counts_all) != "width"]
dim(counts)
counts[1:5, ]
coldata

## 1. Shape of the data
Rows are genes, columns are samples. Note the two things a statistician notices immediately:
- **p ≫ n:** about 19,700 genes but only 10 samples
- the values are non-negative integers spanning several orders of magnitude, with many zeros

In [ ]:
cat("fraction of zero counts:", round(mean(counts == 0), 3), "\n")
cat("genes with zero counts in every sample:", sum(rowSums(counts) == 0), "\n")
hist(log10(counts + 1), breaks = 60, col = "grey80",
     main = "Distribution of counts", xlab = "log10(count + 1)")

## 2. Library size and normalisation
The library size $N_j$ is the total number of reads in sample $j$. If it differs between samples, raw counts are not comparable.

In [ ]:
lib_size <- colSums(counts)
barplot(lib_size / 1e6, las = 2, col = ifelse(coldata$group == "CASE", "tomato", "steelblue"),
        ylab = "library size (millions of reads)", main = "Library sizes")
round(lib_size / 1e6, 1)

🔍 **Sanity check.** A typical RNA-seq sample has 20–50 million reads, but these library sizes are in the **billions**. The counts in this dataset are much larger than one sequencing run produces. Always check that the numbers are plausible before you trust a dataset. For this lecture only the *relative* sizes matter.

**CPM** (counts per million) corrects for library size:
$\text{CPM}_{ij} = K_{ij} / N_j \times 10^6$

**TPM** (transcripts per million) also corrects for gene length $\ell_i$: first divide counts by length, then rescale each sample to sum to $10^6$.

In [ ]:
cpm <- t(t(counts) / lib_size) * 1e6

rpk <- counts / (gene_length / 1000)          # reads per kilobase
tpm <- t(t(rpk) / colSums(rpk)) * 1e6

colSums(cpm)[1:3]   # every sample sums to 1e6
colSums(tpm)[1:3]

✏️ **Exercise 1.** Pick two genes of very different length (look at `gene_length`). Compare their ranks by raw count and by TPM in sample `CTRL_1`. Why does the order change?

In [ ]:
# your code here

## 3. Mean–variance: Poisson or negative binomial?
Within one group (biological replicates), compute each gene's mean and variance of CPM-scaled counts.
If counts were Poisson, points would sit on the line $\text{Var} = \mu$. The negative binomial predicts $\text{Var} = \mu + \phi\mu^2$.

In [ ]:
ctrl <- counts[, coldata$group == "CTRL"]
# scale each control sample to the mean library size, so depth differences don't inflate the variance
sf   <- colSums(ctrl) / mean(colSums(ctrl))
ctrl_n <- t(t(ctrl) / sf)

m <- rowMeans(ctrl_n)
v <- apply(ctrl_n, 1, var)
keep <- m > 1

# method-of-moments estimate of a single dispersion: Var - mu = phi * mu^2
phi_hat <- median(((v - m) / m^2)[keep])
cat("estimated dispersion phi =", round(phi_hat, 3), "\n")

plot(m[keep], v[keep], log = "xy", pch = 16, cex = 0.3, col = rgb(0.4, 0.3, 0.8, 0.3),
     xlab = "mean", ylab = "variance", main = "Mean-variance relationship (controls)")
mu <- 10^seq(0, 8, length.out = 100)
lines(mu, mu, col = "grey40", lwd = 2)
lines(mu, mu + phi_hat * mu^2, col = "red", lwd = 2)
legend("topleft", c("Poisson: Var = mu", "NB: Var = mu + phi mu^2"), col = c("grey40", "red"), lwd = 2, bty = "n")

✏️ **Exercise 2.** Repeat for the `CASE` group. Is the dispersion larger or smaller? Suggest a biological reason. (Hint: are tumours from different patients more or less similar than normal tissue?)

In [ ]:
# your code here

## 4. PCA: do tumours and controls separate?
We take $\log_2(\text{CPM} + 1)$ to tame the huge range, keep the 500 most variable genes, and run PCA. This reduces 500 dimensions to 2 that we can plot.

In [ ]:
logcpm <- log2(cpm + 1)
top <- order(apply(logcpm, 1, var), decreasing = TRUE)[1:500]
pca <- prcomp(t(logcpm[top, ]), scale. = FALSE)
pve <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)

plot(pca$x[, 1], pca$x[, 2], pch = 19, cex = 2,
     col = ifelse(coldata$group == "CASE", "tomato", "steelblue"),
     xlab = paste0("PC1 (", pve[1], "%)"), ylab = paste0("PC2 (", pve[2], "%)"),
     main = "PCA of 10 samples")
text(pca$x[, 1], pca$x[, 2], colnames(counts), pos = 3, cex = 0.7)
legend("topright", c("CASE (metastasised)", "CTRL"), col = c("tomato", "steelblue"), pch = 19, bty = "n")

✏️ **Exercise 3.**
1. Which genes have the largest loadings on PC1 (`pca$rotation[, 1]`)? Look two of them up on NCBI Gene. Do they make sense for colon or liver tissue? (Colorectal cancer often metastasises to the liver.)
2. Repeat the PCA **without** the log transform. What dominates PC1 now, and why?
3. Write two sentences: why is "20,000 genes, 10 samples" a hard setting for statistics?

In [ ]:
# your code here

---
**Further reading:** Akalin, *Computational Genomics with R*, ch. 8 (RNA-seq analysis) · Brown, *Genomes 5*, ch. 12 (transcriptomes).